# 04. Lập mô hình học máy

Notebook này tương ứng **Bước 4: Lập mô hình học máy**.

Bài toán là **Binary Classification**: dự đoán `Churn = Yes/No`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

DEFAULT_ROOT = Path(r"C:\vscode\hoctap\Customer-Churn-Analysis")

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents, DEFAULT_ROOT]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Khong tim thay project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_dataset
from src.data.preprocess import preprocess_pipeline
from src.models.train import train_all_models
from src.models.evaluate import evaluate_models
from src.utils.config import load_project_config

config = load_project_config(PROJECT_ROOT / 'configs' / 'config.yaml')
df = load_dataset(PROJECT_ROOT / config['paths']['raw_data'])
bundle = preprocess_pipeline(df, config)
print(bundle.X_train.shape, bundle.X_test.shape)

(8278, 19) (1409, 19)


## 1. Train nhiều mô hình với full features

Các model được tạo trong `src/models/train.py`, gồm Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, AdaBoost, XGBoost, LightGBM và Voting Ensemble.

In [2]:
trained_models, tuning_summary = train_all_models(bundle.X_train, bundle.y_train, config)
print('Models trained:')
print(list(trained_models.keys()))

Models trained:
['logistic_regression', 'decision_tree', 'random_forest', 'gradient_boosting', 'adaboost', 'xgboost', 'lightgbm', 'notebook_voting_ensemble', 'teacher_voting_ensemble']


## 2. Kiểm tra nhanh kết quả training

Ở notebook này chỉ xem mô hình đã train được chưa. Đánh giá sâu sẽ nằm ở bước 6.

In [3]:
results_df, report_map = evaluate_models(trained_models, bundle.X_test, bundle.y_test)
results_df

,model,accuracy,precision,recall,f1,roc_auc
0,adaboost,0.748048,0.517691,0.743316,0.610318,0.827383
1,teacher_voting_ensemble,0.767211,0.550439,0.671123,0.604819,0.830180
2,random_forest,0.765791,0.548246,0.668449,0.602410,0.827951
3,lightgbm,0.775018,0.568019,0.636364,0.600252,0.823057
4,xgboost,0.763662,0.544662,0.668449,0.600240,0.829439
5,gradient_boosting,0.757275,0.533333,0.684492,0.599532,0.828262
6,notebook_voting_ensemble,0.747339,0.517647,0.705882,0.597285,0.826619
7,logistic_regression,0.736693,0.502814,0.716578,0.590959,0.815531
8,decision_tree,0.751597,0.527149,0.622995,0.571078,0.797295


## Kết luận bước 4

- Dữ liệu sau preprocessing đã được đưa vào nhiều mô hình classification.
- Các model sẽ được so sánh kỹ ở notebook đánh giá.